### AmadeusAPI

In [1]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [2]:
import requests
import os
from datetime import datetime, timedelta

In [3]:
# Your Amadeus credentials
API_KEY = "60xKupG2jdRwmNGPfuRbkodxNE7iuUPl"
API_SECRET = "BlPdAJAo5iXE94r2"

In [4]:
class FlightDealHunter:
    def __init__(self, api_key, api_secret, use_production=False):
        self.api_key = api_key
        self.api_secret = api_secret
        
        # Choose environment
        if use_production:
            self.base_url = "https://api.amadeus.com"
            print("Using PRODUCTION environment (real-time data)")
        else:
            self.base_url = "https://test.api.amadeus.com"
            print("Using TEST environment (cached data)")
        
        self.token = None
        # self.load_transit_rules()
    
    def get_access_token(self):
        """Get OAuth2 token"""
        url = f"{self.base_url}/v1/security/oauth2/token"
        
        headers = {
            "Content-Type": "application/x-www-form-urlencoded"
        }
        
        data = {
            "grant_type": "client_credentials",
            "client_id": self.api_key,
            "client_secret": self.api_secret
        }
        
        response = requests.post(url, headers=headers, data=data)
        self.token = response.json()["access_token"]
        return self.token
    
    def search_flights(self, origin, destination, departure_date, return_date, max_results=3):
        """Search flights using v2 API"""
        url = f"{self.base_url}/v2/shopping/flight-offers"
        
        headers = {
            "Authorization": f"Bearer {self.token}"
        }
        
        params = {
            "originLocationCode": origin,
            "destinationLocationCode": destination,
            "departureDate": departure_date,
            "returnDate": return_date,
            "adults": 1,
            "currencyCode": "GBP",
            "max": max_results
        }
        
        response = requests.get(url, headers=headers, params=params)
        return response.json()


In [5]:
scanner = FlightDealHunter(API_KEY, API_SECRET, use_production=False)

scanner.get_access_token()

results = scanner.search_flights(
    origin="LON",
    destination="SGN",
    departure_date="2025-12-20",
    return_date="2026-01-10"
)
results

Using TEST environment (cached data)


{'meta': {'count': 3,
  'links': {'self': 'https://test.api.amadeus.com/v2/shopping/flight-offers?originLocationCode=LON&destinationLocationCode=SGN&departureDate=2025-12-20&returnDate=2026-01-10&adults=1&currencyCode=GBP&max=3'}},
 'data': [{'type': 'flight-offer',
   'id': '1',
   'source': 'GDS',
   'instantTicketingRequired': False,
   'nonHomogeneous': False,
   'oneWay': False,
   'isUpsellOffer': False,
   'lastTicketingDate': '2025-12-20',
   'lastTicketingDateTime': '2025-12-20',
   'numberOfBookableSeats': 9,
   'itineraries': [{'duration': 'PT21H45M',
     'segments': [{'departure': {'iataCode': 'LHR',
        'terminal': '2',
        'at': '2025-12-20T20:25:00'},
       'arrival': {'iataCode': 'PEK',
        'terminal': '3',
        'at': '2025-12-21T14:25:00'},
       'carrierCode': 'CA',
       'number': '938',
       'aircraft': {'code': '773'},
       'operating': {'carrierCode': 'CA'},
       'duration': 'PT10H',
       'id': '3',
       'numberOfStops': 0,
       'bla